# 01. Introduction to LangChain

**An Overview of the LangChain v1 Framework**


## Learning Objectives

Understand the structure and core components of the LangChain framework.

By the end of this notebook, you will understand:

- The three-layer structure of the LangChain v1 framework
- How the ReAct agent pattern works
- The core components and main APIs
- How to set up and verify the development environment


## 1.1 LangChain Framework Overview

LangChain v1 is an integrated framework for building LLM-based agents. It is organized into three layers, and each layer provides a different level of abstraction.

### Three-Layer Structure

![LangChain three-layer architecture](../assets/images/langchain_3layer.png)

| Layer | Role | Target User |
|--------|------|------------|
| **LangChain** | Core APIs for agent creation (`create_agent`, `tool`, `ChatOpenAI`) | All developers |
| **LangGraph** | Building complex workflows (state graphs, checkpointers, streaming) | Intermediate and advanced developers |
| **Deep Agents** | Prebuilt agents (coding, research, and more) | Fast prototyping |

### Major Changes in LangChain v1

| Item | Previous (v0.x) | Current (v1) |
|------|------------------|--------------|
| Agent creation | `create_react_agent()` | **`create_agent()`** |
| Agent import | `from langchain.agents import ...` (various forms) | **`from langchain.agents import create_agent`** |
| Model initialization | Direct use of `ChatOpenAI(...)` | `init_chat_model()` or `ChatOpenAI(...)` |
| Memory | `ConversationBufferMemory` and similar classes | **`InMemorySaver`** (LangGraph checkpointer) |
| Execution engine | AgentExecutor | **LangGraph graph** (internally) |

### Core Design Philosophy

The central design idea in LangChain v1 is that **every agent runs as a LangGraph graph**. An agent created with `create_agent()` is implemented internally as a LangGraph `StateGraph`, which enables:

- **Streaming**: Real-time responses with the `stream()` method
- **State management**: Conversation history through a checkpointer
- **Extensibility**: Adding custom nodes and edges when needed


## 1.2 The ReAct Agent Pattern

The ReAct (Reasoning + Acting) pattern is the default operating model for LangChain v1 agents. The agent repeatedly runs the following loop:

![ReAct loop — reasoning → action → observation](../assets/images/react_loop.png)

### Key Characteristics

- **Autonomous decisions**: The agent decides for itself whether it should use a tool
- **Multi-step reasoning**: The agent breaks complex tasks into multiple steps
- **Observation-based updates**: The agent observes tool results and chooses its next action


## 1.3 Overview of the Main Components

The table below summarizes the core components of LangChain v1:

| Component | Description | Main APIs |
|----------|-------------|----------|
| **Model** | An LLM or chat model. Acts as the agent's “brain.” | `ChatOpenAI`, `init_chat_model()` |
| **Tools** | Functions the agent can use, such as search, calculation, or API calls | `@tool` decorator, `TavilySearch` |
| **Agent** | The execution unit that combines the model and tools. Internally, it is a LangGraph graph | `create_agent()` |
| **Memory** | A checkpointer that stores and manages conversation history | `InMemorySaver`, `SqliteSaver` |
| **Middleware** | Logic inserted into the request/response processing pipeline | `prompt`, `before_tool`, `after_model` |
| **State** | The state managed while the agent runs, such as messages and context | `AgentState`, `messages` |
| **Streaming** | Support for real-time response streaming | `stream()`, `stream_mode="updates"` |


## 1.4 Environment Setup and Installation Check

Check the packages and API keys required for LangChain v1 development.


In [ ]:
# Environment check
import importlib

packages = {
    "langchain": "langchain",
    "langchain_openai": "langchain-openai",
    "langchain_community": "langchain-community",
    "langgraph": "langgraph",
}

print("=" * 50)
print("LangChain v1 environment check")
print("=" * 50)

for module_name, package_name in packages.items():
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "installed")
        print(f"✓ {package_name}: {version}")
    except ImportError:
        print(f"✗ {package_name}: not installed → pip install {package_name}")


In [ ]:
# API key check
from dotenv import load_dotenv
import os

load_dotenv(override=True)

required_keys = ["OPENAI_API_KEY"]
optional_keys = ["TAVILY_API_KEY", "LANGSMITH_API_KEY"]

print("Required API keys:")
for key in required_keys:
    status = "✓ configured" if os.environ.get(key) else "✗ missing"
    print(f"  {key}: {status}")

print("\nOptional API keys:")
for key in optional_keys:
    status = "✓ configured" if os.environ.get(key) else "- not configured (optional)"
    print(f"  {key}: {status}")


In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: pass config={"callbacks": [langfuse_handler]} to invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
# Verify the core LangChain v1 imports
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

print("✓ Successfully imported all core modules")
print("  - create_agent: create a LangChain v1 agent")
print("  - tool: tool decorator")
print("  - ChatOpenAI: OpenAI-compatible chat model")
print("  - InMemorySaver: memory checkpointer")


## 1.5 Next Steps

In this notebook, you explored the overall structure and core components of the LangChain v1 framework.

In the next notebook (`02_quickstart.ipynb`), you will create and run an actual agent:

- Define a custom tool with the `@tool` decorator
- Create an agent with `create_agent()`
- Run the agent with `invoke()` and `stream()`
- Build a multi-turn conversation with `InMemorySaver`


## Summary

| Item | Description |
|------|-------------|
| LangChain v1 | An agent-centered framework built around the `create_agent()` API |
| Three-layer structure | LangChain → LangGraph → Deep Agents |
| ReAct pattern | Repeated loop of reasoning → action → observation |
| Core APIs | `create_agent()`, `@tool`, `invoke()`, `stream()` |

### Next Steps
→ **[02_quickstart.ipynb](./02_quickstart.ipynb)**: Build your first LangChain agent.
